<a href="https://colab.research.google.com/github/Biplab4/Machine-Learning-Portfolio/blob/main/Deep%20Learning/Cat_Dog_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import shutil
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from google.colab import drive

drive.mount('/content/drive')

print("\nGPU Available:", tf.config.list_physical_devices('GPU'))

zip_path = '/content/drive/MyDrive/CNN_Project/S37 - dataset.zip'
drive_folder = '/content/drive/MyDrive/CNN_Project/dataset'
local_target = '/content/local_dataset'

os.makedirs(local_target, exist_ok=True)

if os.path.exists(zip_path):
    print("\n⚡ Unzipping dataset directly to fast local disk...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_target)
    print("✅ Extraction complete!")
elif os.path.exists(drive_folder):
    print("\n⚡ Found unzipped dataset folder in Drive! Copying locally...")
    shutil.copytree(drive_folder, os.path.join(local_target, 'dataset'), dirs_exist_ok=True)
    print("✅ Copy complete!")
else:
    raise FileNotFoundError("Could not locate dataset zip or folder in Google Drive!")

train_dir = None
val_dir = None

for root, dirs, files in os.walk(local_target):
    if 'training_set' in dirs:
        train_dir = os.path.join(root, 'training_set')
    if 'test_set' in dirs:
        val_dir = os.path.join(root, 'test_set')

if not train_dir or not val_dir:
    raise FileNotFoundError("Could not find 'training_set' and 'test_set' subdirectories!")

print(f"📁 Training Path: {train_dir}")
print(f"📁 Validation Path: {val_dir}")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(160, 160),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(160, 160),
    batch_size=32,
    class_mode='binary'
)

base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


print("\n🚀 Starting Fast GPU Training...\n")
model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

print("\n📦 Exporting model to TFLite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

output_path = '/content/drive/MyDrive/cat_dog_model.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"\n✅ SUCCESS! Download 'cat_dog_model.tflite' directly from your Google Drive root.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

⚡ Found unzipped dataset folder in Drive! Copying locally...
✅ Copy complete!
📁 Training Path: /content/local_dataset/dataset/training_set
📁 Validation Path: /content/local_dataset/dataset/test_set
Found 8000 images belonging to 2 classes.
Found 2001 images belonging to 2 classes.

🚀 Starting Fast GPU Training...

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 103s 334ms/step - accuracy: 0.8576 - loss: 0.3493 - val_accuracy: 0.9300 - val_loss: 0.2040
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 258ms/step - accuracy: 0.9158 - loss: 0.2161 - val_accuracy: 0.9475 - val_loss: 0.1464
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 261ms/step - accuracy: 0.9320 - loss: 0.1782 - val_accuracy: 0.9570 - val_loss: 0.1219
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 64s 257ms/step - accuracy: 0.9402 - loss: